# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [9]:
!pip install -qU langchain langchain-openai python-dotenv pandas

import warnings
warnings.filterwarnings('ignore')

In [11]:

import os
import time
import warnings
import tiktoken

from google.colab import userdata


def load_colab_secret(secret_name: str) -> None:
    """Load a Colab secret into an environment variable."""
    try:
        secret_value = userdata.get(secret_name)

        if not secret_value:
            raise ValueError("Secret is empty.")

        os.environ[secret_name] = secret_value

    except Exception as exc:
        raise RuntimeError(
            f"Could not access {secret_name}. "
            "Make sure it exists in Colab's Secrets tab and "
            "that notebook access is enabled."
        ) from exc


load_colab_secret("OPENAI_API_KEY")
load_colab_secret("PINECONE_API_KEY")
load_colab_secret("HF_TOKEN")

print("Environment ready ✅")

Environment ready ✅


In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [13]:
!pip install pandas

In [14]:
from pathlib import Path

import pandas as pd

data_path = Path('Data.csv')
if not data_path.exists():
    data_path = Path('./Data chains in langchains.csv')
df = pd.read_csv(data_path)

In [15]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

In [16]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [17]:
#Replace None by your own value and justify
llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0.5)


In [19]:
prompt = ChatPromptTemplate.from_template(
    "Write a short and catchy product description for the following product: {product}"
)

In [23]:

chain = prompt | llm | StrOutputParser()

In [24]:
product = df["Product"][0]
chain.invoke(product)

'Sleep like royalty with our Queen Size Sheet Set—ultra-soft, breathable, and designed for a perfect fit. Elevate your bedroom comfort and enjoy nights of luxurious rest!'

## SimpleSequentialChain

In [25]:
llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "Write a short and catchy product description for: {product}"
)

# Chain 1
chain_one = first_prompt | llm | StrOutputParser()

In [26]:

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Based on this product description, suggest 3 target customer types:\n\n{text}"
)
# chain 2
chain_two = second_prompt | llm | StrOutputParser()

In [27]:
overall_simple_chain = chain_one | chain_two

In [29]:
overall_simple_chain.invoke(product)

'1. **Comfort-Seeking Homeowners**  \n   Individuals or families who prioritize high-quality bedding for a cozy and luxurious sleep experience.\n\n2. **Young Professionals**  \n   People setting up their first apartments or upgrading their bedroom essentials with stylish and comfortable sheets.\n\n3. **Gift Buyers**  \n   Shoppers looking for elegant and practical gifts for weddings, housewarmings, or special occasions.'

**Repeat the above twice for different products**

## SequentialChain

In [31]:
llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0.9)


first_prompt = ChatPromptTemplate.from_template(
  "Translate the following review to Italian:\n\n{review}"
)

first_output_key = "italian_review"
chain_one = first_prompt | llm | StrOutputParser()


In [32]:
second_prompt = ChatPromptTemplate.from_template(
    "Summarize this review in 1 sentence:\n\n{italian_review}"
)

second_output_key = "summary"
chain_two = second_prompt | llm | StrOutputParser()


In [33]:
# prompt template 3: translate to italian or other language
third_prompt = ChatPromptTemplate.from_template(
   "What language is this review written in?\n\n{review}"
)
# chain 3: input= Review and output= language
third_output_key = "language"
chain_three = third_prompt | llm | StrOutputParser()


In [34]:

# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
        "Write a polite follow-up email in {language} based on this summary:\n\n{summary}"
)
fourth_output_key = "followup_message"
chain_four = fourth_prompt | llm | StrOutputParser()


In [35]:
# overall_chain: input= Review
# and output= italian_Review,summary, followup_message
input_variables = ["review"]
output_variables = ["italian_review", "summary", "followup_message"]

overall_chain = (
    RunnableLambda(lambda review: {input_variables[0]: review})
    | RunnablePassthrough.assign(**{first_output_key: chain_one})
    | RunnablePassthrough.assign(
        **{second_output_key: chain_two, third_output_key: chain_three}
    )
    | RunnablePassthrough.assign(**{fourth_output_key: chain_four})
    | RunnableLambda(
        lambda state: {key: state[key] for key in [*input_variables, *output_variables]}
    )
)

In [36]:
review = df.Review[5]
overall_chain.invoke(review)

{'review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'italian_review': 'Trovo il gusto mediocre. La schiuma non regge, è strano. Compro gli stessi al supermercato e il gusto è molto migliore...\nVecchio lotto o contraffazione!?',
 'summary': "L'utente ritiene il gusto mediocre e la schiuma scadente rispetto a confezioni precedenti acquistate al supermercato, sospettando un lotto vecchio o una contraffazione.",
 'followup_message': "Objet : Suivi concernant votre avis sur notre produit\n\nBonjour,\n\nNous vous remercions d'avoir pris le temps de nous faire part de votre avis concernant notre produit. Nous sommes désolés d'apprendre que votre expérience n'a pas été à la hauteur de vos attentes, notamment en ce qui concerne le goût et la qualité de la mousse.\n\nAfin de mieux comprendre la situation et de vous apporter une solution adaptée, pourriez-vous s'il vous 

**Repeat the above twice for different products or reviews**

## Router Chain

In [37]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts,
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity.

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [38]:
prompt_infos = [
    {
        "name": "physics",
        "description": "Good for answering questions about physics",
        "prompt_template": physics_template
    },
    {
        "name": "math",
        "description": "Good for answering math questions",
        "prompt_template": math_template
    },
    {
        "name": "History",
        "description": "Good for answering history questions",
        "prompt_template": history_template
    },
    {
        "name": "computer science",
        "description": "Good for answering computer science questions",
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [39]:
from typing import Literal

from pydantic import BaseModel, Field

class RouteQuery(BaseModel):
    destination: Literal[
        'physics', 'math', 'History', 'computer science', 'biology', 'DEFAULT'
    ] = Field(description='The best prompt for the input')
    next_inputs: str = Field(description='The original or improved input')

In [40]:
llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0)

In [41]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = prompt | llm | StrOutputParser()
    destination_chains[name] = chain

destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [42]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = default_prompt | llm | StrOutputParser()

In [43]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT >>"""

In [44]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = ChatPromptTemplate.from_template(router_template)
router_chain = router_prompt | llm.with_structured_output(
    RouteQuery, method='json_schema'
)

In [45]:
def route_input(user_input):
    route = router_chain.invoke({'input': user_input})
    selected_chain = destination_chains.get(route.destination, default_chain)
    return selected_chain.invoke(route.next_inputs)

chain = RunnableLambda(route_input)

In [46]:
chain.invoke("What is black body radiation?")

'Black body radiation is the electromagnetic radiation emitted by an idealized object called a black body, which absorbs all incident radiation regardless of frequency or angle. When this object is at a certain temperature, it emits radiation with a characteristic spectrum that depends only on its temperature, not on its material. This spectrum is continuous and peaks at a wavelength inversely proportional to the temperature (Wien’s law). Black body radiation was crucial in the development of quantum mechanics because classical physics couldn’t explain its observed spectrum, leading to Planck’s introduction of quantized energy levels.'

In [47]:
chain.invoke("what is 2 + 2")

"Great! Let's break down the problem:\n\n1. Identify the numbers involved: 2 and 2.\n2. Understand the operation: addition (+).\n3. Add the two numbers together: 2 + 2.\n\nNow, performing the addition:\n\n2 + 2 = 4.\n\nSo, the answer is **4**."

In [48]:
chain.invoke("Why does every cell in our body contain DNA?")

'Every cell in our body contains DNA because DNA holds the complete set of instructions necessary for the structure, function, and regulation of the organism. Here’s a detailed explanation:\n\n1. **Genetic Blueprint:** DNA (deoxyribonucleic acid) contains genes, which are sequences of nucleotides that encode the information to build proteins and RNA molecules. Proteins are essential for virtually all cellular functions, including metabolism, signaling, structural support, and replication.\n\n2. **Cellular Identity and Function:** Although different cell types (e.g., muscle cells, nerve cells, skin cells) perform specialized functions, they all originate from a single fertilized egg and share the same DNA. The difference in cell types arises from differential gene expression—cells turn on or off specific genes depending on their role, but the underlying DNA sequence remains the same.\n\n3. **Development and Growth:** During development, cells divide and differentiate, but each new cell 

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [50]:
chain.invoke("What causes the Northern Lights?")

"The Northern Lights, or Aurora Borealis, are caused by charged particles from the Sun—mainly electrons and protons—colliding with gases in Earth's atmosphere. These particles are funneled by Earth's magnetic field toward the polar regions, where they interact with oxygen and nitrogen atoms. This interaction excites the atoms, causing them to emit light, which we see as colorful glowing patterns in the sky."